# SIRAP USA — RT-DETRv2-S y OpenVINO Runtime

Notebook de entrenamiento, validación e inferencia del detector de matrículas para Estados Unidos.

El modelo se ajusta previamente con Transformers y PyTorch; la inferencia y la evaluación se ejecutan con OpenVINO Runtime mediante Python. El checkpoint de inicio es merve/license-plates-rtdetrv2, declarado Apache-2.0 y basado en ResNet-18vd. Consulta la ficha del [checkpoint](https://huggingface.co/merve/license-plates-rtdetrv2), el [modelo base](https://huggingface.co/PekingU/rtdetr_v2_r18vd) y la [documentación de RT-DETRv2 en Transformers](https://huggingface.co/docs/transformers/model_doc/rt_detr_v2).

**Para entrenar detección necesitas fotografías completas de vehículos o escenas, con cajas de placas en formato COCO Detection.** Las imágenes recortadas de matrículas que ya están en datasets/usa/seg_and_ocr sirven para evaluar OCR; por sí solas no sirven para entrenar detección en escenas.

Ejecuta las celdas en orden. Las celdas de Optuna, entrenamiento final y pruebas pesadas tienen interruptores para evitar ejecuciones costosas por accidente.

## 0. Instalar dependencias de entrenamiento

Colab normalmente ya incluye PyTorch con CUDA. Esta celda no fija ni reemplaza la versión de PyTorch del entorno.

In [ ]:
import subprocess
import sys

packages = [
    "transformers>=5.4,<6",
    "accelerate>=1.2,<2",
    "pycocotools>=2.0.8",
    "optuna>=4,<5",
    "openvino>=2025.4,<2027",
    "fast-plate-ocr==1.1.0",
    "opencv-python-headless==4.12.0.88",
    "matplotlib>=3.8",
    "pandas>=2.2,<4",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
print("Dependencias instaladas. Si Colab solicita reiniciar el entorno, reinícialo y continúa desde la celda 1.")

## 1. Obtener el repositorio SIRAP_USA_ECUADOR

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPOSITORY_URL = "https://github.com/eduardoceballos666/SIRAP_USA_ECUADOR.git"
preferred = Path("/content/SIRAP_USA_ECUADOR")
candidates = [preferred, Path("/content/SIRAP_USA_ECUADOR_notebook")]

def prepare_clone(path):
    if (path / ".git").is_dir():
        status = subprocess.run(
            ["git", "-C", str(path), "status", "--porcelain"],
            check=True, capture_output=True, text=True,
        ).stdout.strip()
        if status:
            return False
        subprocess.run(
            ["git", "-C", str(path), "pull", "--ff-only", "origin", "main"],
            check=True,
        )
        return True
    if path.exists() and any(path.iterdir()):
        return False
    subprocess.run(["git", "clone", "--depth", "1", REPOSITORY_URL, str(path)], check=True)
    return True

REPO_ROOT = next((path for path in candidates if prepare_clone(path)), None)
if REPO_ROOT is None:
    raise RuntimeError(
        "No encontré una carpeta limpia para el clon. Conservé las carpetas existentes. "
        "Revisa /content/SIRAP_USA_ECUADOR y /content/SIRAP_USA_ECUADOR_notebook."
    )
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
print("Repositorio:", REPO_ROOT)

## 2. Configurar datos y salidas

Cambia DATA_DIR por la raíz COCO de tus escenas de Estados Unidos. Puede estar en Google Drive o en el almacenamiento temporal de Colab.

In [ ]:
import json
import math
import shutil
import time
import hashlib
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

COUNTRY = "usa"
MODEL_ID = "merve/license-plates-rtdetrv2"
IMAGE_SIZE = 640
DATA_DIR = REPO_ROOT / "datasets/usa/detection_coco"
MODEL_OUTPUT = REPO_ROOT / "models/usa/detector/rtdetrv2_s"
METRICS_DIR = REPO_ROOT / "results/metrics"
METRICS_DIR.mkdir(parents=True, exist_ok=True)
RUNS_ROOT = Path("/content/rtdetrv2_usa_runs")
RUNS_ROOT.mkdir(parents=True, exist_ok=True)

GPU_AVAILABLE = torch.cuda.is_available()
GPU_MEMORY_GB = (
    torch.cuda.get_device_properties(0).total_memory / 1024**3
    if GPU_AVAILABLE else 0.0
)
BATCH_SIZE = 8 if GPU_MEMORY_GB >= 16 else 4

print("País:", COUNTRY)
print("Checkpoint:", MODEL_ID)
print("Dispositivo de entrenamiento:", torch.cuda.get_device_name(0) if GPU_AVAILABLE else "CPU")
print(f"Memoria GPU: {GPU_MEMORY_GB:.1f} GB | batch inicial recomendado: {BATCH_SIZE}")
print("Raíz de datos:", DATA_DIR)

## 3. Preparar el dataset COCO

Estructura requerida:

- train/_annotations.coco.json y sus imágenes
- valid/_annotations.coco.json y sus imágenes
- test/_annotations.coco.json y sus imágenes reservadas

También se acepta la carpeta val en lugar de valid. Cada imagen debe tener dimensiones y nombre de archivo correctos; cada caja COCO usa x, y, ancho y alto en píxeles. Usa una sola categoría, license_plate. Incluye escenas sin placas con cero anotaciones.

Separa por cámara, recorrido, vehículo o secuencia antes de crear los splits. No distribuyas cuadros casi idénticos de una misma secuencia entre train, valid y test. Conserva test cerrado hasta congelar hiperparámetros y umbral.

### Crear COCO desde escenas USA anotadas

La plantilla CSV está en datasets/usa/detection_coco/annotations_template.csv. Copia allí las escenas completas en source_images y rellena una fila por cada caja de placa; para una escena sin placas, deja las cuatro coordenadas vacías. source_group debe mantener juntas las imágenes de la misma secuencia, vehículo o sesión. El convertidor verifica rutas, cajas, grupos y duplicados entre splits y genera los JSON COCO que usa el entrenador.

In [ ]:
import shutil
from scripts.usa_annotations_to_coco import convert_annotations

COCO_ROOT = DATA_DIR
SOURCE_IMAGES = COCO_ROOT / "source_images"
ANNOTATION_TEMPLATE = COCO_ROOT / "annotations_template.csv"
ANNOTATIONS_CSV = COCO_ROOT / "annotations.csv"
SOURCE_IMAGES.mkdir(parents=True, exist_ok=True)

if not ANNOTATIONS_CSV.exists() and ANNOTATION_TEMPLATE.is_file():
    shutil.copy2(ANNOTATION_TEMPLATE, ANNOTATIONS_CSV)
    print("Copié la plantilla a", ANNOTATIONS_CSV, "; agrega una fila por caja antes de convertir.")

RUN_COCO_CONVERSION = False
if RUN_COCO_CONVERSION:
    coco_summary = convert_annotations(
        SOURCE_IMAGES, ANNOTATIONS_CSV, COCO_ROOT, verify_hashes=True
    )
    print(coco_summary)
else:
    print("Deja RUN_COCO_CONVERSION en False hasta tener escenas y anotaciones completas.")

In [ ]:
from src.train import CocoPlateDataset, _annotation_file

SPLITS = {}
DATA_READY = False
TEST_READY = False

if not DATA_DIR.is_dir():
    print("No está configurada la carpeta de escenas COCO.")
    print("Cambia DATA_DIR y vuelve a ejecutar esta celda. Las carpetas de recortes OCR no reemplazan las escenas con cajas.")
else:
    for split in ("train", "val", "test"):
        try:
            dataset = CocoPlateDataset(DATA_DIR, split, image_processor=None)
            payload = json.loads(dataset.annotation_path.read_text(encoding="utf-8"))
            categories = payload.get("categories", [])
            missing_images, invalid_boxes = [], []
            for record in dataset.images:
                try:
                    image_path = dataset._image_path(record["file_name"])
                except FileNotFoundError:
                    missing_images.append(record["file_name"])
                    continue
                for ann in dataset.annotations_by_image.get(int(record["id"]), []):
                    x, y, w, h = map(float, ann["bbox"])
                    if w <= 0 or h <= 0 or x < 0 or y < 0:
                        invalid_boxes.append((record["file_name"], ann["bbox"]))
            SPLITS[split] = {
                "dataset": dataset,
                "payload": payload,
                "categories": categories,
                "images": len(dataset.images),
                "boxes": sum(len(v) for v in dataset.annotations_by_image.values()),
                "missing_images": missing_images,
                "invalid_boxes": invalid_boxes,
            }
            print(
                f"{split}: {len(dataset.images)} imágenes, "
                f"{SPLITS[split]['boxes']} cajas, "
                f"{len(categories)} categorías, "
                f"{len(missing_images)} imágenes faltantes, "
                f"{len(invalid_boxes)} cajas inválidas"
            )
        except FileNotFoundError:
            print(f"No se encontró anotación COCO para {split}; es obligatorio train y valid, y test para la evaluación final.")

    DATA_READY = (
        "train" in SPLITS and "val" in SPLITS
        and len(SPLITS["train"]["categories"]) == 1
        and len(SPLITS["val"]["categories"]) == 1
        and not SPLITS["train"]["missing_images"]
        and not SPLITS["val"]["missing_images"]
    )
    TEST_READY = "test" in SPLITS and not SPLITS["test"]["missing_images"]
    print("Datos listos para entrenar:", DATA_READY)
    print("Split test disponible:", TEST_READY)
    if not DATA_READY:
        print("Revisa una sola categoría de placa, los archivos COCO y las rutas de imágenes antes de entrenar.")

## 4. Revisar etiquetas visualmente

Inspecciona cajas pequeñas, placas parcialmente cubiertas, reflejos y escenas negativas. Corrige las cajas antes de entrenar.

In [ ]:
from src.data_processing import load_image

if DATA_READY:
    sample_data = SPLITS["train"]["dataset"]
    sample_count = min(4, len(sample_data.images))
    fig, axes = plt.subplots(1, sample_count, figsize=(5 * sample_count, 4))
    if sample_count == 1:
        axes = [axes]
    for ax, record in zip(axes, sample_data.images[:sample_count]):
        image_path = sample_data._image_path(record["file_name"])
        image = Image.fromarray(load_image(image_path.read_bytes()))
        draw = ImageDraw.Draw(image)
        for ann in sample_data.annotations_by_image.get(int(record["id"]), []):
            x, y, w, h = map(float, ann["bbox"])
            draw.rectangle((x, y, x + w, y + h), outline="lime", width=3)
        ax.imshow(image)
        ax.set_title(record["file_name"])
        ax.axis("off")
    plt.tight_layout()
else:
    print("Configura y valida DATA_DIR antes de revisar ejemplos.")

## 5. Protocolo recomendado de entrenamiento

Punto de partida completo para RT-DETRv2-S en 640 píxeles:

- Ajustar desde merve/license-plates-rtdetrv2; no iniciar desde pesos aleatorios.
- Hasta 60 épocas, batch 8 si la GPU tiene al menos 16 GB y batch 4 en GPU menor; el código detiene temprano si la pérdida de validación no mejora durante cinco evaluaciones.
- Tasa de aprendizaje inicial 5e-5 y weight decay 1e-4. Primero se explora un rango acotado con Optuna; luego se compara con tres semillas.
- El entrenamiento guarda el mejor checkpoint según pérdida de validación. Después, cada candidato se mide con mAP COCO sobre validación y se elige el mejor. Test se usa una sola vez al final.
- Mantén 640 para que la exportación y la app compartan el tamaño validado. Si las placas pequeñas siguen perdiéndose, reúne más escenas de ese tipo antes de cambiar resolución o reglas.

Estos valores son una configuración inicial reproducible, no una garantía de métricas. La distribución, calidad y variedad del dataset determinan el resultado.

In [ ]:
EPOCHS_FINAL = 60
MAX_TRIALS = 8
EPOCHS_TUNING = 12
FINAL_SEEDS = [42, 123, 2026]
DEFAULT_PARAMS = {
    "learning_rate": 5e-5,
    "weight_decay": 1e-4,
    "batch_size": BATCH_SIZE,
}
SELECTED_PARAMS = dict(DEFAULT_PARAMS)
print(SELECTED_PARAMS)

## 6. Búsqueda opcional de hiperparámetros

Activa RUN_OPTUNA cuando train y valid estén listos y dispongas de GPU. La búsqueda compara tasa de aprendizaje, regularización y batch usando solo mAP50-95 de validación. Para un presupuesto mayor, sube MAX_TRIALS gradualmente; no uses test durante la búsqueda.

In [ ]:
from src.train import train_detector
from src.evaluate import evaluate_detector
import optuna

RUN_OPTUNA = False
TUNING_DIR = RUNS_ROOT / "optuna"

def objective(trial):
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-4, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 5e-4, log=True)
    batch_choices = [4, 8] if GPU_MEMORY_GB >= 16 else [4]
    batch_size = trial.suggest_categorical("batch_size", batch_choices)
    output_dir = TUNING_DIR / f"trial_{trial.number}"
    train_detector(
        DATA_DIR, country=COUNTRY, model_id=MODEL_ID,
        output_dir=output_dir, run_dir=output_dir / "trainer",
        epochs=EPOCHS_TUNING, batch_size=batch_size,
        learning_rate=learning_rate, weight_decay=weight_decay,
        seed=42, force=True,
    )
    metrics = evaluate_detector(
        DATA_DIR, country=COUNTRY, split="val",
        checkpoint=output_dir / "openvino", model_id=MODEL_ID,
        threshold=0.001,
        result_path=METRICS_DIR / f"optuna_usa_trial_{trial.number}.json",
    )
    score = metrics.get("map50_95")
    if score is None:
        raise RuntimeError("La evaluación de validación no produjo mAP50-95.")
    return float(score)

if RUN_OPTUNA:
    if not DATA_READY:
        raise RuntimeError("Corrige DATA_DIR antes de iniciar Optuna.")
    study = optuna.create_study(direction="maximize", study_name="rtdetrv2_s_usa")
    study.optimize(objective, n_trials=MAX_TRIALS)
    SELECTED_PARAMS = {
        "learning_rate": study.best_params["learning_rate"],
        "weight_decay": study.best_params["weight_decay"],
        "batch_size": study.best_params["batch_size"],
    }
    print("Mejor configuración de validación:", SELECTED_PARAMS)
else:
    print("Optuna está apagado. Se usarán los valores iniciales hasta que ejecutes la búsqueda.")

## 7. Entrenamiento final con tres semillas

Activa RUN_FINAL_TRAINING después de revisar las etiquetas y, si corresponde, terminar la búsqueda. Cada semilla se entrena por separado; se selecciona el candidato con mayor mAP50-95 en validación. No se promedian los pesos.

In [ ]:
RUN_FINAL_TRAINING = False

if RUN_FINAL_TRAINING:
    if not DATA_READY:
        raise RuntimeError("No se inicia el entrenamiento: falta un dataset COCO válido.")
    candidates = []
    for seed in FINAL_SEEDS:
        run_dir = RUNS_ROOT / f"seed_{seed}"
        train_detector(
            DATA_DIR, country=COUNTRY, model_id=MODEL_ID,
            output_dir=run_dir, run_dir=run_dir / "trainer",
            epochs=EPOCHS_FINAL,
            batch_size=SELECTED_PARAMS["batch_size"],
            learning_rate=SELECTED_PARAMS["learning_rate"],
            weight_decay=SELECTED_PARAMS["weight_decay"],
            seed=seed, force=True,
        )
        metrics = evaluate_detector(
            DATA_DIR, country=COUNTRY, split="val",
            checkpoint=run_dir / "openvino", model_id=MODEL_ID,
            threshold=0.001,
            result_path=METRICS_DIR / f"rtdetr_usa_val_seed_{seed}.json",
        )
        candidates.append({
            "seed": seed,
            "run_dir": str(run_dir),
            "map50_95": metrics.get("map50_95"),
            "map50": metrics.get("map50"),
            "recall": metrics.get("recall"),
            "precision": metrics.get("precision"),
        })

    ranked = [item for item in candidates if item["map50_95"] is not None]
    if not ranked:
        raise RuntimeError("Ningún candidato produjo mAP50-95.")
    best_run = max(ranked, key=lambda item: item["map50_95"])
    source_dir = Path(best_run["run_dir"])
    MODEL_OUTPUT.mkdir(parents=True, exist_ok=True)

    if any(MODEL_OUTPUT.iterdir()):
        from datetime import datetime, timezone
        backup_dir = Path("/content") / f"usa_rtdetr_backup_{datetime.now(timezone.utc):%Y%m%d_%H%M%S}"
        shutil.copytree(MODEL_OUTPUT, backup_dir, dirs_exist_ok=True)
        print("Copia previa conservada en:", backup_dir)

    shutil.copytree(source_dir / "transformers", MODEL_OUTPUT / "transformers", dirs_exist_ok=True)
    shutil.copytree(source_dir / "openvino", MODEL_OUTPUT / "openvino", dirs_exist_ok=True)
    (METRICS_DIR / "rtdetr_usa_three_seed_validation.json").write_text(
        json.dumps({"candidates": candidates, "selected": best_run}, indent=2),
        encoding="utf-8",
    )
    print("Modelo USA seleccionado por validación:", best_run)
    print("IR OpenVINO:", MODEL_OUTPUT / "openvino" / "openvino_model.xml")
else:
    print("Entrenamiento final apagado. Cambia RUN_FINAL_TRAINING a True para iniciar tres corridas.")

## 8. Cargar el modelo con OpenVINO

Si el ajuste fino del país existe, esta celda lo carga. Si todavía no entrenaste, exporta el checkpoint compartido a IR para permitir una prueba de funcionamiento; ese resultado genérico no demuestra desempeño en placas de Estados Unidos.

In [ ]:
from src.export_openvino import export_openvino_model
from src.model import PlateDetector

COUNTRY_IR_DIR = MODEL_OUTPUT / "openvino"
SHARED_IR_DIR = REPO_ROOT / "models/shared/detector/rtdetrv2_s/openvino"
COUNTRY_IR_XML = COUNTRY_IR_DIR / "openvino_model.xml"

if COUNTRY_IR_XML.is_file():
    DETECTOR_DIR = COUNTRY_IR_DIR
    MODEL_IS_COUNTRY_FINETUNED = True
else:
    DETECTOR_DIR = SHARED_IR_DIR
    MODEL_IS_COUNTRY_FINETUNED = False
    if not (SHARED_IR_DIR / "openvino_model.xml").is_file():
        export_openvino_model(MODEL_ID, SHARED_IR_DIR, image_size=IMAGE_SIZE)

detector = PlateDetector(
    checkpoint=DETECTOR_DIR,
    model_id=MODEL_ID,
    imgsz=IMAGE_SIZE,
    conf=0.001,
    class_id=0,
    device="CPU",
    fallback_checkpoint=None,
)
print("Modelo cargado:", detector.name)
print("Ajustado específicamente para USA:", MODEL_IS_COUNTRY_FINETUNED)
print("Dispositivos OpenVINO:", detector.core.available_devices)

## 9. Prueba rápida del IR OpenVINO

In [ ]:
blank_image = np.zeros((480, 640, 3), dtype=np.uint8)
smoke_detections = detector.detect(blank_image, max_detections=20)
print("La inferencia OpenVINO terminó.")
print("Cajas devueltas en la imagen vacía:", len(smoke_detections))
print("Esta comprobación solo confirma carga y ejecución; no mide calidad.")

## 10. Evaluación final de detección

Ejecuta esta celda una vez, después de seleccionar y congelar el modelo. Requiere test COCO y el IR entrenado del perfil USA. Reporta mAP50-95, AP50, AP75, precisión, recall, F1 y métricas por tamaño de objeto.

In [ ]:
RUN_FINAL_TEST = False

if RUN_FINAL_TEST:
    if not TEST_READY:
        raise RuntimeError("Falta el split test COCO reservado.")
    if not COUNTRY_IR_XML.is_file():
        raise RuntimeError("Falta el IR ajustado para USA. Entrena el modelo antes de evaluar.")
    final_test_metrics = evaluate_detector(
        DATA_DIR, country=COUNTRY, split="test",
        checkpoint=COUNTRY_IR_DIR, model_id=MODEL_ID,
        threshold=0.001,
        result_path=METRICS_DIR / "detector_rtdetr_usa_test.json",
    )
    display(pd.DataFrame([final_test_metrics]))
else:
    print("Test final apagado. Actívalo solo cuando el modelo y el umbral estén congelados.")

## 11. Calibrar el umbral con validación

Esta celda no usa test. Calcula precisión, recall y F1 en IoU 0.50 para varios umbrales de confianza. Se prioriza el recall entre los umbrales que cumplen la precisión objetivo; si ninguno la cumple, se muestra el mejor F1 como referencia provisional.

In [ ]:
def box_iou_xyxy(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)
    area_a = max(0.0, a[2] - a[0]) * max(0.0, a[3] - a[1])
    area_b = max(0.0, b[2] - b[0]) * max(0.0, b[3] - b[1])
    return inter / max(area_a + area_b - inter, 1e-9)

def collect_detection_records(split, max_images=None, transform=None):
    dataset = CocoPlateDataset(DATA_DIR, split, image_processor=None)
    source_images = dataset.images if max_images is None else dataset.images[:max_images]
    records = []
    for item in source_images:
        image_path = dataset._image_path(item["file_name"])
        image = load_image(image_path.read_bytes())
        if transform is not None:
            image = transform(image)
        gt_boxes = []
        for ann in dataset.annotations_by_image.get(int(item["id"]), []):
            x, y, w, h = map(float, ann["bbox"])
            gt_boxes.append((x, y, x + w, y + h))
        detections = detector.detect(image, max_detections=300)
        records.append({
            "image_id": int(item["id"]),
            "predictions": [(d.box, d.confidence) for d in detections],
            "ground_truth": gt_boxes,
        })
    return records

def score_detection_records(records, threshold, iou_threshold=0.50):
    tp = fp = fn = 0
    for record in records:
        unmatched = set(range(len(record["ground_truth"])))
        predictions = sorted(
            (item for item in record["predictions"] if item[1] >= threshold),
            key=lambda item: item[1], reverse=True,
        )
        for box, _ in predictions:
            best_index, best_iou = None, 0.0
            for index in unmatched:
                value = box_iou_xyxy(box, record["ground_truth"][index])
                if value > best_iou:
                    best_index, best_iou = index, value
            if best_index is not None and best_iou >= iou_threshold:
                tp += 1
                unmatched.remove(best_index)
            else:
                fp += 1
        fn += len(unmatched)
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)
    return {"tp": tp, "fp": fp, "fn": fn, "precision": precision, "recall": recall, "f1": f1}

RUN_THRESHOLD_CALIBRATION = False
TARGET_PRECISION = 0.99
if RUN_THRESHOLD_CALIBRATION:
    if not DATA_READY or not COUNTRY_IR_XML.is_file():
        raise RuntimeError("Necesitas train, valid y un IR ajustado de USA para calibrar.")
    val_records = collect_detection_records("val")
    threshold_rows = [
        {"threshold": round(float(value), 2), **score_detection_records(val_records, float(value))}
        for value in np.arange(0.05, 1.00, 0.05)
    ]
    threshold_table = pd.DataFrame(threshold_rows)
    eligible = threshold_table[threshold_table["precision"] >= TARGET_PRECISION]
    if len(eligible):
        selected = eligible.sort_values(["recall", "threshold"], ascending=[False, True]).iloc[0]
        selection_rule = "máximo recall sujeto a precisión objetivo"
    else:
        selected = threshold_table.sort_values("f1", ascending=False).iloc[0]
        selection_rule = "mejor F1; la precisión objetivo no se alcanzó"
    selected_threshold = float(selected["threshold"])
    ACTIVE_THRESHOLD = selected_threshold
    result = {
        "country": COUNTRY,
        "model": "RT-DETRv2-S",
        "runtime": "OpenVINO Runtime Python",
        "split": "valid",
        "iou_threshold": 0.50,
        "target_precision": TARGET_PRECISION,
        "selection_rule": selection_rule,
        "threshold": selected_threshold,
        "metrics": {key: float(selected[key]) for key in ("precision", "recall", "f1")},
    }
    (METRICS_DIR / "calibracion_umbral_rtdetr_usa.json").write_text(
        json.dumps(result, indent=2), encoding="utf-8"
    )
    display(threshold_table)
    print("Umbral candidato de validación:", result)
else:
    print("Calibración apagada. Activa RUN_THRESHOLD_CALIBRATION después del entrenamiento.")

## 12. Pruebas de robustez de imagen

Compara el test reservado sin alteraciones con perturbaciones sintéticas de iluminación, desenfoque, reducción de tamaño y compresión. Son pruebas de estrés reproducibles; no sustituyen escenas reales de día, noche, lluvia, reflejos, ángulos y distancia.

In [ ]:
import cv2

def darker(image):
    return np.clip(image.astype(np.float32) * 0.55, 0, 255).astype(np.uint8)

def brighter(image):
    return np.clip(image.astype(np.float32) * 1.25 + 12, 0, 255).astype(np.uint8)

def blurred(image):
    return cv2.GaussianBlur(image, (5, 5), 0)

def reduced(image):
    h, w = image.shape[:2]
    small = cv2.resize(image, (max(1, w // 2), max(1, h // 2)))
    return cv2.resize(small, (w, h), interpolation=cv2.INTER_LINEAR)

def compressed(image):
    bgr = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    ok, encoded = cv2.imencode(".jpg", bgr, [cv2.IMWRITE_JPEG_QUALITY, 35])
    if not ok:
        return image
    decoded = cv2.imdecode(encoded, cv2.IMREAD_COLOR)
    return cv2.cvtColor(decoded, cv2.COLOR_BGR2RGB)

RUN_ROBUSTNESS_TEST = False
ACTIVE_THRESHOLD = 0.25
if RUN_ROBUSTNESS_TEST:
    if not TEST_READY or not COUNTRY_IR_XML.is_file():
        raise RuntimeError("Necesitas test COCO y el IR ajustado del perfil USA.")
    threshold_file = METRICS_DIR / "calibracion_umbral_rtdetr_usa.json"
    if threshold_file.is_file():
        ACTIVE_THRESHOLD = json.loads(threshold_file.read_text(encoding="utf-8"))["threshold"]
    variants = {
        "original": None,
        "baja_iluminacion": darker,
        "alta_iluminacion": brighter,
        "desenfoque": blurred,
        "reduccion_50_por_ciento": reduced,
        "compresion_jpeg": compressed,
    }
    stress_rows = []
    for name, transform in variants.items():
        records = collect_detection_records("test", max_images=200, transform=transform)
        stress_rows.append({"condition": name, **score_detection_records(records, ACTIVE_THRESHOLD)})
    robustness_table = pd.DataFrame(stress_rows)
    display(robustness_table)
    robustness_table.to_csv(METRICS_DIR / "robustez_imagen_rtdetr_usa.csv", index=False)
else:
    print("Prueba de estrés apagada. No cambia el modelo ni el umbral de producción.")

## 13. Latencia de inferencia OpenVINO en CPU

Mide por separado inferencia del detector sobre imágenes test, con calentamiento previo. El resultado mide únicamente el detector, sin OCR, lectura de archivos ni red.

In [ ]:
RUN_LATENCY_TEST = False
if RUN_LATENCY_TEST:
    if not TEST_READY or not COUNTRY_IR_XML.is_file():
        raise RuntimeError("Necesitas test COCO y el IR ajustado del perfil USA.")
    test_dataset = SPLITS["test"]["dataset"]
    samples = test_dataset.images[:min(200, len(test_dataset.images))]
    for item in samples[:10]:
        image = load_image(test_dataset._image_path(item["file_name"]).read_bytes())
        detector.detect(image, max_detections=100)
    latencies_ms = []
    for item in samples[10:]:
        image = load_image(test_dataset._image_path(item["file_name"]).read_bytes())
        start = time.perf_counter()
        detector.detect(image, max_detections=100)
        latencies_ms.append((time.perf_counter() - start) * 1000)
    latency_summary = {
        "n": len(latencies_ms),
        "device": "CPU",
        "median_ms": float(np.median(latencies_ms)) if latencies_ms else None,
        "p95_ms": float(np.percentile(latencies_ms, 95)) if latencies_ms else None,
        "p99_ms": float(np.percentile(latencies_ms, 99)) if latencies_ms else None,
        "fps_from_median": 1000 / float(np.median(latencies_ms)) if latencies_ms else None,
    }
    (METRICS_DIR / "latencia_openvino_usa.json").write_text(
        json.dumps(latency_summary, indent=2), encoding="utf-8"
    )
    print(latency_summary)
else:
    print("Medición de latencia apagada.")

## 14. Evaluar OCR en recortes de matrículas de EE. UU.

Esta medición es independiente de detección. El CSV debe contener una fila por imagen con nombre de archivo, estado y placa real; las imágenes se leen desde datasets/usa/seg_and_ocr/usimages.

In [ ]:
RUN_OCR_CROP_TEST = False
OCR_CSV = REPO_ROOT / "datasets/usa/seg_and_ocr/groundtruth.csv"
OCR_IMAGE_DIR = REPO_ROOT / "datasets/usa/seg_and_ocr/usimages"

if RUN_OCR_CROP_TEST:
    from fast_plate_ocr import LicensePlateRecognizer
    from src.data_processing import clean_text, plate_distance

    if not OCR_CSV.is_file() or not OCR_IMAGE_DIR.is_dir():
        raise FileNotFoundError("No encontré el CSV o los recortes OCR del perfil USA.")
    ocr_truth = pd.read_csv(
        OCR_CSV, header=None, names=["image_file", "state", "plate_gt"], dtype=str
    ).dropna(subset=["image_file", "plate_gt"])
    recognizer = LicensePlateRecognizer("cct-s-v2-global-model", device="cpu")
    ocr_rows = []
    for row in ocr_truth.itertuples(index=False):
        image_path = OCR_IMAGE_DIR / row.image_file
        if not image_path.is_file():
            continue
        crop = np.asarray(Image.open(image_path).convert("RGB"))
        prediction = recognizer.run(crop, return_confidence=True)[0]
        pred = clean_text(prediction.plate)
        gt = clean_text(row.plate_gt)
        probs = np.asarray(prediction.char_probs, dtype=np.float32).ravel()
        ocr_rows.append({
            "image_file": row.image_file,
            "state": row.state,
            "plate_gt": gt,
            "plate_pred": pred,
            "exact_match": pred == gt,
            "char_errors": plate_distance(pred, gt),
            "gt_characters": len(gt),
            "confidence_min": float(probs.min()) if probs.size else 0.0,
        })
    ocr_results = pd.DataFrame(ocr_rows)
    emr = float(ocr_results["exact_match"].mean()) if len(ocr_results) else 0.0
    cer = (
        float(ocr_results["char_errors"].sum() / max(ocr_results["gt_characters"].sum(), 1))
        if len(ocr_results) else 0.0
    )
    print({"n": len(ocr_results), "exact_match_rate": emr, "character_error_rate": cer})
    display(ocr_results.groupby("state")["exact_match"].agg(["count", "mean"]).sort_values("mean"))
    ocr_results.to_csv(METRICS_DIR / "ocr_recortes_rtdetr_usa.csv", index=False)
else:
    print("Evaluación de OCR en recortes apagada.")

## 15. Prueba punta a punta en escenas completas

Actívala con un CSV independiente que tenga image_path y plate_gt. Opcionalmente incluye state. Se mide detección más OCR y se informan imágenes sin caja; no se aplican reglas de acceso ni normalización específica por país.

In [ ]:
RUN_SCENE_E2E_TEST = False
SCENE_E2E_CSV = REPO_ROOT / "datasets/usa/scene_e2e_test.csv"

if RUN_SCENE_E2E_TEST:
    from fast_plate_ocr import LicensePlateRecognizer
    from src.data_processing import clean_text, crop_with_margin, plate_distance

    if not COUNTRY_IR_XML.is_file():
        raise RuntimeError("Entrena y congela primero el IR USA.")
    if not SCENE_E2E_CSV.is_file():
        raise FileNotFoundError("Crea scene_e2e_test.csv con columnas image_path y plate_gt.")
    scene_truth = pd.read_csv(SCENE_E2E_CSV, dtype=str).dropna(subset=["image_path", "plate_gt"])
    if not {"image_path", "plate_gt"}.issubset(scene_truth.columns):
        raise ValueError("El CSV necesita image_path y plate_gt.")
    recognizer = LicensePlateRecognizer("cct-s-v2-global-model", device="cpu")
    predictions = []
    for row in scene_truth.itertuples(index=False):
        candidate_path = Path(row.image_path)
        if not candidate_path.is_absolute():
            options = [REPO_ROOT / candidate_path, DATA_DIR / candidate_path]
            candidate_path = next((p for p in options if p.is_file()), options[0])
        image = load_image(candidate_path.read_bytes())
        detections = [
            d for d in detector.detect(image, max_detections=20)
            if d.confidence >= ACTIVE_THRESHOLD
        ]
        candidates = []
        for detection in detections:
            crop = crop_with_margin(image, detection.box, 0.06)
            if crop.size == 0:
                continue
            read = recognizer.run(crop, return_confidence=True)[0]
            text = clean_text(read.plate)
            probs = np.asarray(read.char_probs, dtype=np.float32).ravel()
            confidence = float(probs.mean()) if probs.size else 0.0
            candidates.append((text, confidence, detection.confidence))
        best = max(candidates, key=lambda item: (item[1], item[2])) if candidates else ("", 0.0, 0.0)
        gt = clean_text(row.plate_gt)
        predictions.append({
            "image_path": str(candidate_path),
            "plate_gt": gt,
            "plate_pred": best[0],
            "exact_match": best[0] == gt,
            "char_errors": plate_distance(best[0], gt),
            "detected": bool(detections),
            "det_confidence": best[2],
            "ocr_confidence": best[1],
        })
    e2e_results = pd.DataFrame(predictions)
    print({
        "n": len(e2e_results),
        "exact_match_rate": float(e2e_results["exact_match"].mean()) if len(e2e_results) else 0.0,
        "character_error_rate": float(
            e2e_results["char_errors"].sum() / max(e2e_results["plate_gt"].str.len().sum(), 1)
        ) if len(e2e_results) else 0.0,
        "missed_detection_rate": float((~e2e_results["detected"]).mean()) if len(e2e_results) else 0.0,
    })
    e2e_results.to_csv(METRICS_DIR / "e2e_imagenes_rtdetr_usa.csv", index=False)
    display(e2e_results.head())
else:
    print("Prueba punta a punta apagada.")

## 16. Lista de validación antes de integrar en decisiones de acceso

- Dataset representativo: distintos estados y diseños, día/noche, reflejos, lluvia, ángulos, distancia, placas pequeñas y oclusiones.
- Sin fuga entre particiones: separar por vehículo, cámara y secuencia; revisar imágenes repetidas.
- Detector: mAP50-95, AP50, AP75, precisión, recall, F1, falsos positivos y falsos negativos; desglosar placas pequeñas.
- Umbral: fijarlo con validación según el costo de falsos positivos; no ajustarlo mirando test.
- Robustez: comparar escenas reales por condición además de las perturbaciones sintéticas.
- OCR separado y punta a punta: reportar coincidencia exacta, errores de caracteres y fallas de detección, incluidos resultados por estado.
- CPU: reportar mediana y percentil 95 con el mismo tamaño de entrada y hardware del Space.
- Acceso: antes de automatizar una barrera, medir falsas autorizaciones y falsos rechazos con una lista y ground truth independientes; las lecturas dudosas deben pasar a revisión humana.

El notebook no ejecuta decisiones de acceso para Estados Unidos. El checkpoint compartido solo sirve como punto de partida hasta completar el ajuste y la validación con escenas locales.

## 17. Publicar el IR USA a GitHub y al Space

Después de revisar las métricas de test, activa la publicación. Requiere el secreto GITHUB_TOKEN de Colab con permiso de escritura en SIRAP_USA_ECUADOR. El flujo publica el IR del país; GitHub Actions lo sincroniza con el Space.

In [ ]:
PUBLISH_IR_TO_SPACE = False
if PUBLISH_IR_TO_SPACE:
    from scripts.notebook_io import publish_detector_ir
    if not COUNTRY_IR_XML.is_file():
        raise FileNotFoundError("Primero entrena y exporta el modelo USA.")
    print(publish_detector_ir(REPO_ROOT, COUNTRY))
else:
    print("Publicación apagada.")